<a href="https://colab.research.google.com/github/rony22mansor/BaseProject/blob/main/cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.model_selection import RepeatedKFold
import json
from lightgbm import LGBMRegressor

In [ ]:
COLS_TO_DROP = [
    'ID', 'title', 'make', 'model', 'year', 'vin', 'scrape_timestamp','feature_count',
    'year_title', 'year_vin', 'year_final', 'features_list', 'trans', 'title_status',
    'search_price', 'ext_color', 'int_color', 'est._monthly_pmt', 'trim','engine_cleaned', 'features_list_cleaned',
]

df_test_model = df_test.drop(columns=[c for c in COLS_TO_DROP if c in df_test.columns], errors='ignore')
test_ids = df_test['ID'].copy()
for col in df_test_model.columns:
    if df_test_model[col].dtype == 'object':
        df_test_model[col] = df_test_model[col].fillna('unknown').astype(str).astype("category")

COLS_TO_DROP = [c for c in COLS_TO_DROP if c in df_train.columns]
df_model = df_train.drop(columns=COLS_TO_DROP)
for col in df_model.columns:
    if df_model[col].dtype == 'object':
        df_model[col] = df_model[col].fillna('unknown').astype(str).astype("category")

In [ ]:
with open("/content/drive/MyDrive/ML-Car_Price_Prediction_Challenge_2026/dataset/best_lightgbm_params.json", "r") as f:
    best_params = json.load(f)
print(best_params)

y = np.log1p(df_model["price"])
X = df_model.drop(columns=["price"])
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = (
            X[col]
            .fillna("unknown")
            .astype(str)
            .astype("category")
        )

cat_features = X.select_dtypes(
    include=["category"]
).columns.tolist()

best_model = LGBMRegressor(
    **best_params,
    objective="regression",
    metric="rmse",
    random_state=42,
    n_jobs=-1
)


{'min_split_gain': 6.086595038812259e-05, 'learning_rate': 0.006048517747584449, 'n_estimators': 3393, 'num_leaves': 186, 'max_depth': 5, 'min_child_samples': 16, 'subsample': 0.8149350611814493, 'colsample_bytree': 0.9364919131257943, 'reg_alpha': 0.013508394381274498, 'max_bin': 251, 'min_data_in_bin': 5, 'reg_lambda': 1.0421614746010937, 'feature_fraction_bynode': 0.6007227984659633, 'feature_fraction': 0.6714460042675511, 'min_child_weight': 0.02052012979642307}


In [ ]:
selected_features = [
    col for col in X.columns
    if col not in [ "model_final"]
]

X_selected = X[selected_features]
df_test_selected = df_test_model[selected_features]

cat_features_selected = [
    c for c in cat_features
    if c in selected_features
]

best_model.fit(
    X_selected,
    y,
    categorical_feature=cat_features_selected
)
preds = best_model.predict(df_test_selected)
final_preds = np.expm1(preds)
submission = pd.DataFrame({
    "ID": test_ids,
    "price": final_preds
})

submission.to_csv(
    "submission_selected_features.csv",
    index=False
)

print(submission.head())
importance = pd.DataFrame({
    "Feature": X_selected.columns,
    "Importance": best_model.booster_.feature_importance(
        importance_type="gain"
    )
}).sort_values("Importance", ascending=False)

print(importance)

[LightGBM] [Warning] feature_fraction is set=0.6714460042675511, colsample_bytree=0.9364919131257943 will be ignored. Current value: feature_fraction=0.6714460042675511
[LightGBM] [Warning] feature_fraction is set=0.6714460042675511, colsample_bytree=0.9364919131257943 will be ignored. Current value: feature_fraction=0.6714460042675511
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.055686 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 559
[LightGBM] [Info] Number of data points in the train set: 775, number of used features: 26
[LightGBM] [Info] Start training from score 9.189846
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

## Ensampling

In [ ]:
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor


In [ ]:
import numpy as np
import pandas as pd
import optuna
import warnings

optuna.logging.set_verbosity(optuna.logging.WARNING)

from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score

y        = np.log1p(df_train['price'])
X        = df_train.drop(columns=['price']).copy()
X_test   = df_test.copy()

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=['float64','int64']).columns.tolist()

imputer = SimpleImputer(strategy='median')
X[num_cols]      = imputer.fit_transform(X[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])

for col in cat_cols:
    X[col]      = X[col].fillna('unknown').astype(str)
    X_test[col] = X_test[col].fillna('unknown').astype(str)

KF_TUNE = KFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial):
    params = dict(
        iterations          = trial.suggest_int('iterations', 500, 3000, step=100),
        learning_rate       = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        depth               = trial.suggest_int('depth', 3, 8),
        l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1, 20),
        bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 2.0),
        random_strength     = trial.suggest_float('random_strength', 0.0, 2.0),
        border_count        = trial.suggest_int('border_count', 32, 254),
        loss_function       = 'RMSE',
        eval_metric         = 'RMSE',
        random_seed         = 42,
        thread_count        = -1,
        verbose             = 0,
    )

    scores = []
    for tr_idx, val_idx in KF_TUNE.split(X):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        tr_pool  = Pool(X_tr,  y_tr,  cat_features=cat_cols)
        val_pool = Pool(X_val, y_val, cat_features=cat_cols)

        model = CatBoostRegressor(**params)
        model.fit(tr_pool, eval_set=val_pool,
                  early_stopping_rounds=50, use_best_model=True)

        preds = np.expm1(model.predict(X_val))
        true  = np.expm1(y_val)
        scores.append(r2_score(true, preds))

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
best_params.update({
    'loss_function' : 'RMSE',
    'eval_metric'   : 'RMSE',
    'random_seed'   : 42,
    'thread_count'  : -1,
    'verbose'       : 0,
})

KF      = KFold(n_splits=5, shuffle=True, random_state=42)
oof     = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(KF.split(X), 1):
    X_tr,  X_val  = X.iloc[tr_idx],  X.iloc[val_idx]
    y_tr,  y_val  = y.iloc[tr_idx],  y.iloc[val_idx]

    tr_pool   = Pool(X_tr,     y_tr,  cat_features=cat_cols)
    val_pool  = Pool(X_val,    y_val, cat_features=cat_cols)
    test_pool = Pool(X_test,          cat_features=cat_cols)

    model = CatBoostRegressor(**best_params)
    model.fit(tr_pool, eval_set=val_pool,
              early_stopping_rounds=100, use_best_model=True)

    oof[val_idx] = model.predict(X_val)
    test_preds  += model.predict(test_pool) / 5

    r2 = r2_score(np.expm1(y_val), np.expm1(oof[val_idx]))
    fold_scores.append(r2)
    print(f"  Fold {fold}  R = {r2:.4f}  (best iter: {model.best_iteration_})")

overall_r2 = r2_score(np.expm1(y), np.expm1(oof))

print(f"  CV  R = {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"  OOF R = {overall_r2:.4f}")


importance = pd.DataFrame({
    'Feature'    : X.columns,
    'Importance' : model.get_feature_importance()
}).sort_values('Importance', ascending=False)


print(importance.head(15).to_string(index=False))

final_prices = np.clip(np.expm1(test_preds), 0, None)

submission = pd.DataFrame({
    'ID'   : test_ids,
    'price': final_prices
})
submission.to_csv('submission_catboost.csv', index=False)

print(f"   Price range: ${final_prices.min():,.0f} → ${final_prices.max():,.0f}")
print(f"   Price mean : ${final_prices.mean():,.0f}")
print(submission.head())

Train : (784, 25)
Test  : (278, 25)

Categorical (16): ['condition', 'cylinders', 'drive', 'fuel', 'paint_color', 'title_status', 'transmission', 'type', 'make_final', 'model_final', 'extracted_trim', 'engine_source', 'engine_layout', 'engine', 'features_list', 'engine_text']
Numeric     (9): ['odometer_scaled', 'car_age', 'is_turbo', 'is_electric_engine', 'is_luxury', 'engine_liters', 'engine_cylinders', 'is_electric', 'is_hybrid']

NaNs in X    : 0
NaNs in X_test: 0
✅ Data ready

🔍 Running Optuna (50 trials × 3-fold) — ~5 min...


  0%|          | 0/50 [00:00<?, ?it/s]


✅ Best Optuna R² = 0.2376
   Best params    = {'iterations': 1200, 'learning_rate': 0.24341582739801756, 'depth': 5, 'l2_leaf_reg': 2.03283759142555, 'bagging_temperature': 1.5586094833582766, 'random_strength': 0.9270294866665691, 'border_count': 187, 'loss_function': 'RMSE', 'eval_metric': 'RMSE', 'random_seed': 42, 'thread_count': -1, 'verbose': 0}

🔁 Final 5-Fold Training with best params
  Fold 1  R² = -0.0099  (best iter: 49)
  Fold 2  R² = 0.2532  (best iter: 22)
  Fold 3  R² = 0.5394  (best iter: 68)
  Fold 4  R² = 0.2053  (best iter: 19)
  Fold 5  R² = 0.4298  (best iter: 42)

  CV  R² = 0.2836 ± 0.1897
  OOF R² = -0.0018   ← most reliable estimate

📊 Top 15 Features:
         Feature  Importance
         car_age   27.511350
 odometer_scaled   18.038617
           drive    7.649666
       cylinders    6.703098
  extracted_trim    6.083071
            type    5.556560
            fuel    4.328846
   engine_liters    3.702625
     paint_color    3.636567
   engine_layout    3.2

In [ ]:
y = np.log1p(df_train['price'])

X     = df_train.drop(columns=['price']).copy()
X_test = df_test.copy()

cat_cols = X.select_dtypes(include='object').columns.tolist()
print(f'Train shape   : {X.shape}')
print(f'Test shape    : {X_test.shape}')
print(f'Categorical   : {cat_cols}')
print(f'Target (y) mean: {y.mean():.4f}  std: {y.std():.4f}')

📊 Train shape   : (784, 25)
📊 Test shape    : (278, 16)
📌 Categorical   : ['condition', 'cylinders', 'drive', 'fuel', 'paint_color', 'title_status', 'transmission', 'type', 'make_final', 'model_final', 'extracted_trim', 'engine_source', 'engine_layout', 'engine', 'features_list', 'engine_text']
🎯 Target (y) mean: 9.1601  std: 1.1593


In [ ]:
X_lgb      = X.copy()
X_test_lgb = X_test.copy()

for col in cat_cols:
    X_lgb[col]      = X_lgb[col].fillna('unknown').astype(str)
    X_test_lgb[col] = X_test_lgb[col].fillna('unknown').astype(str)
    cats = sorted(set(X_lgb[col].unique()) | set(X_test_lgb[col].unique()))
    X_lgb[col]      = pd.Categorical(X_lgb[col], categories=cats)
    X_test_lgb[col] = pd.Categorical(X_test_lgb[col], categories=cats)

from sklearn.preprocessing import OrdinalEncoder

X_xgb      = X.copy()
X_test_xgb = X_test.copy()

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)


✅ All data versions ready


In [ ]:

with open('/content/best_lightgbm_params.json') as f:
    lgb_params = json.load(f)

lgb_params = dict(
    n_estimators      = 2000,
    learning_rate     = 0.05,
    num_leaves        = 31,
    max_depth         = 5,
    min_child_samples = 10,
    feature_fraction  = 0.70,
    bagging_fraction  = 0.70,
    bagging_freq      = 1,
    reg_alpha         = 1.0,
    reg_lambda        = 5.0,
    objective         = 'regression',
    metric            = 'rmse',
    random_state      = 42,
    n_jobs            = -1,
    verbosity         = -1,
)




In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()

num_imputer = SimpleImputer(strategy='median')
X[num_cols]      = num_imputer.fit_transform(X[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

X_lgb      = X.copy()
X_test_lgb = X_test.copy()

for col in cat_cols:
    X_lgb[col]      = X_lgb[col].fillna('unknown').astype(str)
    X_test_lgb[col] = X_test_lgb[col].fillna('unknown').astype(str)

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_lgb[cat_cols]      = enc.fit_transform(X_lgb[cat_cols])
X_test_lgb[cat_cols] = enc.transform(X_test_lgb[cat_cols])




In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X))


test_preds_lgb = np.zeros(len(X_test))

lgb_scores, xgb_scores, cat_scores = [], [], []


for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f'\n FOLD {fold}/{N_FOLDS}')

    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    X_tr_lgb  = X_lgb.iloc[train_idx]
    X_val_lgb = X_lgb.iloc[val_idx]

    X_tr_xgb  = X_xgb.iloc[train_idx]
    X_val_xgb = X_xgb.iloc[val_idx]

    X_tr_cat  = X_cat.iloc[train_idx]
    X_val_cat = X_cat.iloc[val_idx]

    lgb_model = LGBMRegressor(**lgb_params)
    lgb_model.fit(
        X_tr_lgb, y_tr,
        eval_set=[(X_val_lgb, y_val)],
        callbacks=[
            __import__('lightgbm').early_stopping(stopping_rounds=100, verbose=False),
            __import__('lightgbm').log_evaluation(period=-1)
        ],
        categorical_feature=cat_cols,
    )
    oof_lgb[val_idx]    = lgb_model.predict(X_val_lgb)
    test_preds_lgb     += lgb_model.predict(X_test_lgb) / N_FOLDS
    r2_lgb = r2_score(np.expm1(y_val), np.expm1(oof_lgb[val_idx]))
    lgb_scores.append(r2_lgb)


print('\n' + '='*55)
print('CROSS-VALIDATION SUMMARY')
print(f'  LightGBM  CV R² = {np.mean(lgb_scores):.4f} ± {np.std(lgb_scores):.4f}')


🔁 Starting 5-Fold Cross-Validation

📂 FOLD 1/5
  LightGBM  R² = -0.0087  (best iter: 67)

📂 FOLD 2/5
  LightGBM  R² = 0.5766  (best iter: 288)

📂 FOLD 3/5
  LightGBM  R² = 0.6272  (best iter: 98)

📂 FOLD 4/5
  LightGBM  R² = 0.2648  (best iter: 40)

📂 FOLD 5/5
  LightGBM  R² = 0.4061  (best iter: 377)

📊 CROSS-VALIDATION SUMMARY
  LightGBM  CV R² = 0.3732 ± 0.2300
